# Pipeline ML — Prédiction du risque d'annulation
**Maeva — Mastère 2 DIA Paris**

Pipeline complet :
1. Chargement & nettoyage
2. Feature engineering (variables dérivées)
3. Gestion du déséquilibre (SMOTE + class_weight)
4. Préprocessing (imputation, scaling, OHE)
5. Entraînement de 3 modèles (LR → RF → XGBoost)
6. Optimisation des hyperparamètres (RandomizedSearchCV)
7. Évaluation complète (ROC, PR, matrice de confusion)
8. Interprétabilité SHAP
9. Export du modèle final

---

## 0. Installation

In [ ]:
# %pip install pandas numpy matplotlib seaborn scikit-learn xgboost shap imbalanced-learn joblib

## 1. Imports & configuration

In [ ]:
import os, re, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
warnings.filterwarnings('ignore')

# Préprocessing
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Gestion déséquilibre
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

# Modèles
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Métriques
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    average_precision_score, precision_recall_curve,
    f1_score, precision_score, recall_score
)
from sklearn.metrics import make_scorer

# Interprétabilité
import shap

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
os.makedirs('figures', exist_ok=True)
os.makedirs('models',  exist_ok=True)

RANDOM_STATE = 42
print('✅ Imports OK')

## 2. Chargement & filtres

In [ ]:
CSV_PATH = 'data/dataset_annulation_anon.csv'
df = pd.read_csv(CSV_PATH, encoding='utf-8-sig', low_memory=False)
print(f'Chargé : {df.shape[0]:,} lignes × {df.shape[1]} colonnes')

# ── Filtre périmètre Maeva ───────────────────────────────────────
conditions_maeva = ['Flexi', 'Flexi+', 'NoFlex', 'NoFlex (<J30)']
df = df[df['cond_annulation'].isin(conditions_maeva)].copy().reset_index(drop=True)

# ── Vérification de la cible ─────────────────────────────────────
n_total = len(df)
n_annul = df['y_annulation'].sum()
taux    = n_annul / n_total * 100

print(f'\nAprès filtre Maeva : {n_total:,} dossiers')
print(f'  Y=0 (maintenu) : {n_total - n_annul:,} ({100-taux:.1f}%)')
print(f'  Y=1 (annulé)   : {n_annul:,} ({taux:.1f}%)')
print(f'  Ratio déséquilibre : 1 annulation pour {(n_total-n_annul)/n_annul:.1f} maintiens')

## 3. Feature engineering

In [ ]:
# ── Plafonnement des outliers de fidélité ────────────────────────
if 'nb_dossiers_anterieurs' in df.columns:
    df['nb_dossiers_anterieurs'] = df['nb_dossiers_anterieurs'].clip(upper=10)

# ── Features dérivées ────────────────────────────────────────────

# a) Anticipation en tranches (capte les non-linéarités mieux que le continu)
if 'anticipation_jours' in df.columns:
    df['anticipation_tranche'] = pd.cut(
        df['anticipation_jours'],
        bins=[-1, 7, 30, 90, 180, 99999],
        labels=['derniere_minute', 'court_30j', 'moyen_90j', 'long_180j', 'tres_long']
    ).astype(str)

# b) Interaction assurance × anticipation
#    L'effet de la couverture Flex dépend-il du délai avant le séjour ?
if {'est_assure_annulation', 'anticipation_jours'}.issubset(df.columns):
    df['assure_x_anticip'] = (
        df['est_assure_annulation'] * df['anticipation_jours'].clip(upper=365)
    )

# c) Voyageur solo (1 adulte, 0 mineur)
if {'dossier_nb_pax_adultes', 'nb_mineur'}.issubset(df.columns):
    df['est_solo'] = (
        (df['dossier_nb_pax_adultes'] == 1) & (df['nb_mineur'] == 0)
    ).astype(int)

# d) Réservation en week-end (DAYOFWEEK BigQuery : 1=Dim, 6=Ven, 7=Sam)
if 'jour_semaine_resa' in df.columns:
    df['resa_weekend'] = df['jour_semaine_resa'].isin([1, 6, 7]).astype(int)

# ── Aperçu rapide des nouvelles features ─────────────────────────
new_feats = ['anticipation_tranche', 'assure_x_anticip', 'est_solo', 'resa_weekend']
for f in new_feats:
    if f in df.columns:
        grp = df.groupby(f)['y_annulation'].mean().sort_values(ascending=False)
        print(f'\n{f} — taux annulation :')
        print((grp*100).round(2))

## 4. Définition des features & split train/test

In [ ]:
EXCLURE = [
    'dossier_cle', 'client_email', 'date_resa',
    'dossier_etat', 'dossier_annule', 'y_annulation',
    'est_dans_crm', 'canal_detail', 'cgv_annul', 'fournisseur_nom',
    'a_promo',   # variable leakage — définitivement exclue
    'jour_semaine_resa',  # remplacée par resa_weekend
]

FEATURES_NUM = [
    'anticipation_jours', 'duree_sejour',
    'dossier_nb_pax_total', 'dossier_nb_pax_adultes', 'nb_mineur', 'nb_bebe',
    'mois_resa',
    'nb_dossiers_anterieurs', 'est_client_vip',
    'nb_clics_90j', 'nb_ouvertures_90j', 'nb_desabo_90j',
    'nb_campagnes_recues', 'nb_campagnes_cliquees',
    'nb_urls_distinctes_cliquees', 'taux_clic_sur_ouverture',
    'a_interagi_email', 'recence_email_jours',
    'est_assure_annulation',
    'assure_x_anticip', 'est_solo', 'resa_weekend',
]

FEATURES_CAT = [
    'canal', 'groupe_fournisseur', 'periode_depart', 'periode_vacances',
    'type_produit', 'device_resa', 'theme_station', 'region_destination',
    'type_hebergement', 'segment_email', 'cond_annulation',
    'anticipation_tranche',
]

# Filtrer aux colonnes présentes
FEATURES_NUM = [c for c in dict.fromkeys(FEATURES_NUM) if c in df.columns and c not in EXCLURE]
FEATURES_CAT = [c for c in dict.fromkeys(FEATURES_CAT) if c in df.columns and c not in EXCLURE]

# Limiter la cardinalité (max 20 modalités par variable catégorielle)
def limiter_cardinalite(data, cols, top_n=20):
    data = data.copy()
    for c in cols:
        top = data[c].value_counts().nlargest(top_n).index
        data[c] = data[c].where(data[c].isin(top), other='autre').astype(str)
    return data

df = limiter_cardinalite(df, FEATURES_CAT)

X = df[FEATURES_NUM + FEATURES_CAT]
y = df['y_annulation'].astype(int)

# Split stratifié 70/30 (stratify garantit le même ratio Y=1 dans train et test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y
)

print(f'Features numériques    : {len(FEATURES_NUM)}')
print(f'Features catégorielles : {len(FEATURES_CAT)}')
print(f'\nTrain : {len(X_train):,} | Y=1 : {y_train.mean()*100:.2f}%')
print(f'Test  : {len(X_test):,}  | Y=1 : {y_test.mean()*100:.2f}%')

## 5. Gestion du déséquilibre

Deux stratégies combinées :
- **SMOTE** sur le train set (génère des exemples synthétiques de la classe minoritaire)
- **class_weight / scale_pos_weight** dans les modèles (pénalise les erreurs sur la classe minoritaire)

> ⚠️ SMOTE s'applique UNIQUEMENT sur X_train — jamais sur le test set (qui reste réel).

In [ ]:
# ── Préprocessing (séparé du pipeline pour SMOTE) ────────────────
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='inconnu')),
    ('ohe',     OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])
preprocessor = ColumnTransformer([
    ('num', num_pipe, FEATURES_NUM),
    ('cat', cat_pipe, FEATURES_CAT)
])

# Préprocessing sur train & test
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep  = preprocessor.transform(X_test)

# Récupération des noms de features après OHE
cat_names = (preprocessor
             .named_transformers_['cat']
             .named_steps['ohe']
             .get_feature_names_out(FEATURES_CAT).tolist())
FEAT_NAMES = FEATURES_NUM + cat_names
FEAT_NAMES_CLEAN = [re.sub(r'[\[\]<>]', '_', n) for n in FEAT_NAMES]

print(f'Features après OHE : {len(FEAT_NAMES)}')
print(f'Train avant SMOTE  : {X_train_prep.shape[0]:,} | Y=1 : {y_train.sum():,} ({y_train.mean()*100:.1f}%)')

# ── SMOTE sur le train uniquement ───────────────────────────────
# Objectif : rééquilibrer à 30% de Y=1 (pas 50% — trop artificiel)
smote = SMOTE(
    sampling_strategy=0.30,   # ratio final Y=1 / Y=0 = 0.30
    random_state=RANDOM_STATE,
    k_neighbors=5
)
X_train_sm, y_train_sm = smote.fit_resample(X_train_prep, y_train)

print(f'\nTrain après SMOTE  : {X_train_sm.shape[0]:,} | Y=1 : {y_train_sm.sum():,} ({y_train_sm.mean()*100:.1f}%)')
print(f'Test (inchangé)    : {X_test_prep.shape[0]:,}  | Y=1 : {y_test.sum():,} ({y_test.mean()*100:.1f}%)')

# scale_pos_weight pour XGBoost (sur les données SMOTE)
spw = (y_train_sm == 0).sum() / (y_train_sm == 1).sum()
print(f'\nscale_pos_weight XGBoost : {spw:.2f}')

## 6. Entraînement des 3 modèles

In [ ]:
def evaluer(nom, model, Xtr, ytr, Xte, yte, seuil=0.5):
    """Entraîne, prédit et retourne toutes les métriques."""
    t0 = time.time()
    model.fit(Xtr, ytr)
    proba = model.predict_proba(Xte)[:, 1]
    pred  = (proba >= seuil).astype(int)

    res = {
        'nom'      : nom,
        'model'    : model,
        'proba'    : proba,
        'pred'     : pred,
        'auc'      : roc_auc_score(yte, proba),
        'ap'       : average_precision_score(yte, proba),
        'f1'       : f1_score(yte, pred),
        'precision': precision_score(yte, pred),
        'recall'   : recall_score(yte, pred),
        'duree'    : round(time.time() - t0, 1)
    }
    print(f"\n{'='*50}\n  {nom} ({res['duree']}s)\n{'='*50}")
    print(f"  AUC-ROC   : {res['auc']:.4f}")
    print(f"  PR-AUC    : {res['ap']:.4f}   (référence = {yte.mean():.4f})")
    print(f"  F1        : {res['f1']:.4f}")
    print(f"  Précision : {res['precision']:.4f}")
    print(f"  Rappel    : {res['recall']:.4f}")
    return res

results = []

# ── Modèle 1 : Régression Logistique (baseline interprétable) ───
lr = LogisticRegression(
    max_iter=2000,
    class_weight='balanced',   # compense le déséquilibre résiduel
    C=0.1,                     # régularisation L2
    solver='lbfgs',
    random_state=RANDOM_STATE
)
results.append(evaluer('Régression Logistique', lr, X_train_sm, y_train_sm, X_test_prep, y_test))

# ── Modèle 2 : Random Forest ─────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=20,
    class_weight='balanced',
    n_jobs=-1,
    random_state=RANDOM_STATE
)
results.append(evaluer('Random Forest', rf, X_train_sm, y_train_sm, X_test_prep, y_test))

# ── Modèle 3 : XGBoost ───────────────────────────────────────────
xgb = XGBClassifier(
    n_estimators=400,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    gamma=0.1,
    reg_alpha=0.01,
    reg_lambda=1.5,
    scale_pos_weight=spw,      # gestion déséquilibre XGBoost
    eval_metric='aucpr',       # optimisé sur PR-AUC
    n_jobs=-1,
    random_state=RANDOM_STATE
)
results.append(evaluer('XGBoost', xgb, X_train_sm, y_train_sm, X_test_prep, y_test))

## 7. Comparaison des modèles

In [ ]:
# ── Tableau récapitulatif ─────────────────────────────────────────
df_res = pd.DataFrame([{
    'Modèle'    : r['nom'],
    'AUC-ROC'   : round(r['auc'], 4),
    'PR-AUC'    : round(r['ap'],  4),
    'F1'        : round(r['f1'],  4),
    'Précision' : round(r['precision'], 4),
    'Rappel'    : round(r['recall'], 4)
} for r in results]).sort_values('PR-AUC', ascending=False).reset_index(drop=True)

print('=== COMPARAISON DES 3 MODÈLES ===')
print(f"Référence PR-AUC (aléatoire) : {y_test.mean():.4f}")
print(df_res.to_string(index=False))

In [ ]:
# ── Courbes ROC + Précision-Rappel ───────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
colors = ['#3498db', '#e67e22', '#2ecc71']

for r, c in zip(results, colors):
    # ROC
    fpr, tpr, _ = roc_curve(y_test, r['proba'])
    ax1.plot(fpr, tpr, color=c, lw=2,
             label=f"{r['nom']} (AUC={r['auc']:.3f})")
    # Précision-Rappel
    prec, rec, _ = precision_recall_curve(y_test, r['proba'])
    ax2.plot(rec, prec, color=c, lw=2,
             label=f"{r['nom']} (AP={r['ap']:.3f})")

ax1.plot([0,1],[0,1],'k--',lw=1,label='Aléatoire (0.5)')
ax1.set(xlabel='Faux positifs', ylabel='Vrais positifs', title='Courbe ROC')
ax1.legend(loc='lower right')

ax2.axhline(y_test.mean(), ls='--', c='k', lw=1,
            label=f'Référence ({y_test.mean():.3f})')
ax2.set(xlabel='Rappel', ylabel='Précision', title='Courbe Précision-Rappel')
ax2.legend(loc='upper right')

plt.tight_layout()
plt.savefig('figures/01_courbes_roc_pr.png', dpi=150)
plt.show()

In [ ]:
# ── Matrice de confusion — XGBoost ───────────────────────────────
best_base = [r for r in results if r['nom'] == 'XGBoost'][0]
cm = confusion_matrix(y_test, best_base['pred'])

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues',
            xticklabels=['Prédit maintenu', 'Prédit annulé'],
            yticklabels=['Réel maintenu',   'Réel annulé'], ax=ax)
ax.set_title('Matrice de confusion — XGBoost')
plt.tight_layout()
plt.savefig('figures/02_confusion.png', dpi=150)
plt.show()

print(classification_report(y_test, best_base['pred'],
      target_names=['Maintenu (0)', 'Annulé (1)']))

## 8. Optimisation des hyperparamètres (XGBoost)
**RandomizedSearchCV** — scoré sur PR-AUC, validation croisée 3-fold.
> Durée estimée : 10-20 min

In [ ]:
param_dist = {
    'n_estimators'     : [300, 400, 500, 600],
    'max_depth'        : [4, 5, 6, 7, 8],
    'learning_rate'    : [0.01, 0.03, 0.05, 0.08, 0.1],
    'subsample'        : [0.6, 0.7, 0.8, 0.9],
    'colsample_bytree' : [0.6, 0.7, 0.8, 0.9],
    'min_child_weight' : [1, 3, 5, 10, 20],
    'gamma'            : [0, 0.05, 0.1, 0.2, 0.5],
    'reg_alpha'        : [0, 0.01, 0.05, 0.1],
    'reg_lambda'       : [1, 1.5, 2, 3],
}

xgb_cv = XGBClassifier(
    scale_pos_weight=spw,
    eval_metric='aucpr',
    n_jobs=-1,
    random_state=RANDOM_STATE
)

cv_strat = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
scorer   = make_scorer(average_precision_score, needs_proba=True)

print('Lancement RandomizedSearchCV (n_iter=40, cv=3, score=PR-AUC)...')
t0 = time.time()

rs = RandomizedSearchCV(
    xgb_cv, param_dist,
    n_iter=40,
    cv=cv_strat,
    scoring=scorer,
    refit=True,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1
)
rs.fit(X_train_sm, y_train_sm)

print(f'\n✅ Terminé en {(time.time()-t0)/60:.1f} min')
print(f'Meilleurs paramètres :')
for k, v in rs.best_params_.items():
    print(f'  {k:22s}: {v}')
print(f'\nMeilleur PR-AUC (CV) : {rs.best_score_:.4f}')

In [ ]:
# ── Évaluation du modèle optimisé ────────────────────────────────
best_opt  = rs.best_estimator_
proba_opt = best_opt.predict_proba(X_test_prep)[:, 1]
pred_opt  = (proba_opt >= 0.5).astype(int)

auc_opt = roc_auc_score(y_test, proba_opt)
ap_opt  = average_precision_score(y_test, proba_opt)
f1_opt  = f1_score(y_test, pred_opt)
rec_opt = recall_score(y_test, pred_opt)
prec_opt = precision_score(y_test, pred_opt)

xgb_base_res = [r for r in results if r['nom'] == 'XGBoost'][0]

print('='*55)
print(f'  {"Métrique":<20} {"XGBoost base":>14} {"XGBoost opt":>12} {"Gain":>8}')
print('='*55)
for nom, base, opt in [
    ('PR-AUC',    xgb_base_res['ap'],        ap_opt),
    ('AUC-ROC',   xgb_base_res['auc'],       auc_opt),
    ('F1-Score',  xgb_base_res['f1'],        f1_opt),
    ('Rappel',    xgb_base_res['recall'],    rec_opt),
    ('Précision', xgb_base_res['precision'], prec_opt),
]:
    print(f'  {nom:<20} {base:>14.4f} {opt:>12.4f} {opt-base:>+8.4f}')
print('='*55)

## 9. Interprétabilité SHAP — XGBoost optimisé

In [ ]:
# Échantillon (max 3 000 pour la vitesse)
Xte_df     = pd.DataFrame(X_test_prep, columns=FEAT_NAMES_CLEAN)
n_shap     = min(3000, len(Xte_df))
Xte_sample = Xte_df.sample(n_shap, random_state=RANDOM_STATE)

best_opt.get_booster().feature_names = None   # évite l'erreur sur [ ] <
explainer  = shap.TreeExplainer(best_opt)
shap_vals  = explainer.shap_values(Xte_sample)
print(f'SHAP calculé sur {n_shap:,} dossiers')

In [ ]:
# Beeswarm — impact et direction
plt.figure()
shap.summary_plot(shap_vals, Xte_sample, max_display=15, show=False)
plt.title('SHAP — Impact des features (XGBoost optimisé)')
plt.tight_layout()
plt.savefig('figures/03_shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Bar plot — importance moyenne
plt.figure()
shap.summary_plot(shap_vals, Xte_sample, plot_type='bar', max_display=15, show=False)
plt.title('SHAP — Importance moyenne (XGBoost optimisé)')
plt.tight_layout()
plt.savefig('figures/04_shap_bar.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Waterfall — explication d'un dossier individuel
idx = 0   # change cet index pour explorer d'autres dossiers
shap.plots.waterfall(
    shap.Explanation(
        values=shap_vals[idx],
        base_values=explainer.expected_value,
        data=Xte_sample.iloc[idx],
        feature_names=FEAT_NAMES_CLEAN
    ),
    max_display=12
)
plt.title(f'SHAP Waterfall — Dossier #{idx}')
plt.tight_layout()
plt.savefig('figures/05_shap_waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Synthèse & export du modèle final

In [ ]:
print('='*55)
print('  SYNTHÈSE PIPELINE ML')
print('='*55)
print(f'  Dataset          : {len(df):,} dossiers')
print(f'  Taux annulation  : {y.mean()*100:.2f}%')
print(f'  SMOTE            : train rééquilibré à {y_train_sm.mean()*100:.0f}% Y=1')
print(f'  Features totales : {len(FEATURES_NUM)+len(FEATURES_CAT)} → {len(FEAT_NAMES)} après OHE')
print()
print(f'  {"Modèle":<25} {"PR-AUC":>8} {"AUC-ROC":>8} {"Rappel":>8}')
print('  ' + '-'*51)
for r in results:
    print(f'  {r["nom"]:<25} {r["ap"]:>8.4f} {r["auc"]:>8.4f} {r["recall"]:>8.4f}')
print(f'  {"XGBoost optimisé":<25} {ap_opt:>8.4f} {auc_opt:>8.4f} {rec_opt:>8.4f}  ← RETENU')
print('='*55)

In [ ]:
# ── Sauvegarde ───────────────────────────────────────────────────
joblib.dump(best_opt,      'models/xgb_final.joblib')
joblib.dump(preprocessor,  'models/preprocessor.joblib')
joblib.dump({
    'features_num'   : FEATURES_NUM,
    'features_cat'   : FEATURES_CAT,
    'feat_names_ohe' : FEAT_NAMES,
    'smote_ratio'    : 0.30,
    'best_params'    : rs.best_params_,
    'pr_auc_opt'     : ap_opt,
    'auc_roc_opt'    : auc_opt,
}, 'models/model_config.joblib')

print('✅ Modèle exporté dans ./models/')
print('  xgb_final.joblib')
print('  preprocessor.joblib')
print('  model_config.joblib')

In [ ]:
# ── Test de rechargement ─────────────────────────────────────────
m = joblib.load('models/xgb_final.joblib')
p = joblib.load('models/preprocessor.joblib')

sample     = X_test.iloc[[0]]
sample_pre = p.transform(sample)
score      = m.predict_proba(sample_pre)[0][1]

print(f'✅ Rechargement OK')
print(f'   Score du dossier test n°0 : {score:.4f} ({score*100:.1f}% de risque d\'annulation)')